In [1]:
import pandas as pd

df = pd.read_csv("../source.csv")
df.head(10)

,userName,content,score,at,appVersion
0,Yuga Edit,akun gopay saya di blok,1,2022-01-21 10:52:12,4.9.3
1,ff burik,Lambat sekali sekarang ini bosssku apk gojek g...,3,2021-11-30 15:40:38,4.9.3
2,Anisa Suci Rahmayuliani,Kenapa sih dari kemarin sy buka aplikasi gojek...,4,2021-11-29 22:58:12,4.9.3
3,naoki yakuza,Baru download gojek dan hape baru trus ditop u...,1,2022-09-03 15:21:17,4.9.3
4,Trio Sugianto,Mantap,5,2022-01-15 10:05:27,4.9.3
5,Arlan Ramlan,Bagus,4,2022-02-01 05:50:40,4.9.3
6,Slamet Hariyanto,Coba dulu,2,2021-12-10 22:40:45,4.9.3
7,Hasan Thio,Ok,5,2022-02-01 03:07:45,4.9.3
8,RAFI BADZLIN,Gimana ini kak pin saya salah terus padahal ud...,1,2022-12-17 08:56:52,4.9.3
9,mariyadi qc,Biar aman kamu tidak bisa pakai gojek Jadi say...,1,2022-02-09 11:27:38,4.9.3


## Text Preprocessing Pipeline

Setup text preprocessing using indoNLP library

In [2]:
from indoNLP.preprocessing import pipeline, emoji_to_words, replace_word_elongation
from indoNLP.preprocessing import SLANG_DATA, SLANG_PATTERN
import re

# Create a safer version of replace_slang that handles missing keys
def safe_replace_slang(text):
    """Replace slang words, but keep original if not found in dictionary"""
    return re.sub(SLANG_PATTERN, lambda mo: SLANG_DATA.get(mo.group(0).lower(), mo.group(0)), text)

# Use safe_replace_slang instead of the library's version
pipe = pipeline([safe_replace_slang, emoji_to_words, replace_word_elongation])

print("✓ Text preprocessing pipeline ready")

✓ Text preprocessing pipeline ready


## Data Cleaning

Remove duplicates and missing values

In [3]:
# Check data before cleaning
print(f"Original data shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Remove rows with missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print(f"\nAfter cleaning:")
print(f"Cleaned data shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

Original data shape: (225002, 5)
Missing values:
userName      0
content       2
score         0
at            0
appVersion    0
dtype: int64
Duplicate rows: 0

After cleaning:
Cleaned data shape: (225000, 5)
Missing values: 0
Duplicate rows: 0


## Create Labels and Final Dataset

Create labels based on score and prepare `source_final.csv`

In [4]:
df['label'] = df['score'].apply(lambda x: 'positive' if x > 3 else 'negative')

df_final = df[['content', 'label', 'score']].copy()

# Clean df_final: remove NA and duplicates
print(f'df_final before cleaning: {len(df_final)} rows')
print(f"Missing values:\n{df_final.isnull().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

df_final = df_final.dropna()
df_final = df_final.drop_duplicates()
df_final = df_final.reset_index(drop=True)

print(f'\ndf_final after cleaning: {len(df_final)} rows')
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicate rows: {df_final.duplicated().sum()}")

# Display label distribution
print(f'\nLabel distribution:')
print(df_final['label'].value_counts())
print(f'\nFirst 10 rows:')
df_final.head(10)

df_final before cleaning: 225000 rows
Missing values:
content    0
label      0
score      0
dtype: int64
Duplicate rows: 83998

df_final after cleaning: 141002 rows
Missing values: 0
Duplicate rows: 0

Label distribution:
label
positive    79447
negative    61555
Name: count, dtype: int64

First 10 rows:


,content,label,score
0,akun gopay saya di blok,negative,1
1,Lambat sekali sekarang ini bosssku apk gojek g...,negative,3
2,Kenapa sih dari kemarin sy buka aplikasi gojek...,positive,4
3,Baru download gojek dan hape baru trus ditop u...,negative,1
4,Mantap,positive,5
5,Bagus,positive,4
6,Coba dulu,negative,2
7,Ok,positive,5
8,Gimana ini kak pin saya salah terus padahal ud...,negative,1
9,Biar aman kamu tidak bisa pakai gojek Jadi say...,negative,1


## Apply Text Preprocessing

Clean and normalize text content

In [5]:
print("Applying text preprocessing to 'content' column...")
print(f"Total rows to process: {len(df_final)}")

# Apply preprocessing pipeline
df_final['content'] = df_final['content'].apply(pipe)

print("✓ Text preprocessing complete!")
print("\nSample of preprocessed content:")
df_final.head(10)

Applying text preprocessing to 'content' column...
Total rows to process: 141002
✓ Text preprocessing complete!

Sample of preprocessed content:


,content,label,score
0,akun gopay saya di blok,negative,1
1,Lambat sekali sekarang ini bosssku apk gojek e...,negative,3
2,Kenapa sih dari kemarin saya buka aplikasi goj...,positive,4
3,Baru download gojek dan hape baru terus ditop ...,negative,1
4,Mantap,positive,5
5,Bagus,positive,4
6,Coba dulu,negative,2
7,Ok,positive,5
8,bagaimana ini kak pin saya salah terus padahal...,negative,1
9,Biar aman kamu tidak bisa pakai gojek Jadi say...,negative,1


In [6]:
df_final.to_csv('../source_final.csv', index=False)
print('Successfully created source_final.csv!')

Successfully created source_final.csv!


## Create Balanced Dataset

Create `source_balanced.csv` with 30,000 positive and 30,000 negative samples

In [7]:
# Check current label distribution
print("Current label distribution:")
print(df_final['label'].value_counts())
print(f"\nTotal samples: {len(df_final)}")

# Sample 30k positive and 30k negative
positive_samples = df_final[df_final['label'] == 'positive'].sample(n=30000, random_state=42)
negative_samples = df_final[df_final['label'] == 'negative'].sample(n=30000, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_samples, negative_samples], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset:")
print(f"Total rows: {len(df_balanced)}")
print(f"\nLabel distribution:")
print(df_balanced['label'].value_counts())
df_balanced.head(10)

Current label distribution:
label
positive    79447
negative    61555
Name: count, dtype: int64

Total samples: 141002

Balanced dataset:
Total rows: 60000

Label distribution:
label
positive    30000
negative    30000
Name: count, dtype: int64


,content,label,score
0,Pengiriman cepat banget,positive,5
1,Jalan masuk ke dalam perumahan bellacasa depok...,negative,1
2,Kenapa apl gojeku sering banget kaya enggak bi...,negative,2
3,Aplikasinya sangat membantu sekali dan gampang...,positive,5
4,Alhamdulillah sellu mendapatkan driver yang ba...,positive,5
5,makin lama makin serakah aplikasi anak bangsa ...,negative,1
6,Keren banyak diskon,positive,5
7,Bagus amat deh ini,positive,5
8,enggak bisa didownload,positive,5
9,aturan promo nya berubah dibeda2in di hp orang...,negative,1


In [8]:
# Save balanced dataset to CSV
df_balanced.to_csv('../source_balanced.csv', index=False)